# check

> Listen back with Su-śrotā: what a render says, how far it is from the text, the best of a few seeds, and word timings.

In [ ]:
#| default_exp check

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

[Su-śrotā](https://huggingface.co/prathoshap/sushrota-sanskrit-asr) is IndicConformer fine-tuned on śāstric Sanskrit. Its CTC head covers all 22
IndicConformer languages; Sanskrit is the 256 columns from 4096, with the blank at 5632. The checkpoint's config names AI4Bharat's NeMo fork
(`multilingual` tokenizer, `multisoftmax` decoder), so `Ear` rewrites it for upstream NeMo: the tokenizer becomes `agg`, the fork-only keys go, and
the weights load non-strict. Only the encoder and the CTC head are used. Install with `pip install vachaka[check]`.

In [ ]:
#| export
import re, tarfile, unicodedata, numpy as np
from fastcore.utils import *
from vachaka.core import cache, clean, concat, Recitation

In [ ]:
#| export
ASR, ASR_FILE = 'prathoshap/sushrota-sanskrit-asr', 'sushrota_sanskrit_asr_v13b.nemo'
OFF, V, BL = 4096, 256, 5632

## Comparing text

In [ ]:
#| export
_DROP = re.compile('[\u0966-\u096F।॥ऽॐ\u0951-\u0954\u1CD0-\u1CFF]')
_KEEP = re.compile('[^\u0900-\u097F\\s]')
def norm(s):
    "Text as Su-śrotā writes it: NFC, no accents, daṇḍas or avagraha, word-final म् as anusvāra, doubled marks collapsed."
    s = _KEEP.sub(' ', _DROP.sub(' ', unicodedata.normalize('NFC', s)))
    s = re.sub('(.)(?=\\1)', lambda m: '' if unicodedata.category(m[1]) in ('Mn', 'Mc') else m[1], s)
    s = re.sub('म्(?=\\s|$)', 'ं', s)
    return re.sub('\\s+', ' ', s).strip()

In [ ]:
test_eq(norm('चोदनालक्षणोऽर्थो धर्मः ॥ २ ॥'), 'चोदनालक्षणो र्थो धर्मः')
test_eq(norm('कंसचाणूरमर्दनम्'), 'कंसचाणूरमर्दनं')

In [ ]:
#| export
def lev(a, b):
    "Edit distance between sequences `a` and `b`."
    row = list(range(len(b)+1))
    for i,x in enumerate(a, 1):
        prev, row[0] = row[0], i
        for j,y in enumerate(b, 1): prev, row[j] = row[j], min(row[j]+1, row[j-1]+1, prev+(x!=y))
    return row[-1]

def cer(ref, hyp):
    "Character error rate of `hyp` against `ref`, both normalised, spaces ignored (sandhi spacing varies)."
    r, h = [norm(o).replace(' ', '') for o in (ref, hyp)]
    return lev(r, h)/max(1, len(r))

In [ ]:
test_eq(lev('kitten', 'sitting'), 3)
test_eq(cer('अथातो धर्मजिज्ञासा', 'अथातो धर्मजिज्ञासा'), 0)
assert 0 < cer('तस्य निमित्तपरीष्टिः', 'तस्य निमित्तपरेष्टिः') < 0.1

## Ear

In [ ]:
#| export
def _cfg(nemo):
    "Su-śrotā's config rewritten for upstream NeMo, cached beside the vachaka sources."
    from omegaconf import OmegaConf, DictConfig
    p = cache/'sushrota_v13b.yaml'
    if p.exists(): return p
    with tarfile.open(nemo) as t: c = OmegaConf.create(t.extractfile('./model_config.yaml').read().decode())
    def strip(d):
        if not isinstance(d, DictConfig): return
        for k in list(d.keys()):
            if k in ('multisoftmax', 'multilingual', 'language_keys', 'return_language_id'): del d[k]
            else: strip(d[k])
    strip(c); c.tokenizer.type = 'agg'
    p.parent.mkdir(parents=True, exist_ok=True); OmegaConf.save(c, p)
    return p

In [ ]:
#| export
class Ear:
    "Su-śrotā on the CPU: posteriors, transcripts and forced alignment."
    def __init__(self):
        import logging, nemo.collections.asr as na
        from huggingface_hub import hf_hub_download
        logging.getLogger('nemo_logger').setLevel(logging.ERROR)
        p = hf_hub_download(ASR, ASR_FILE)
        self.m = na.models.EncDecHybridRNNTCTCBPEModel.restore_from(p, map_location='cpu', override_config_path=str(_cfg(p)), strict=False).eval()
        self.tok = self.m.tokenizer.tokenizers_dict['sa']

    def post(self, audio, sr):
        "Log-posteriors, frames × (blank + 256 Sanskrit tokens), and the ms per frame."
        import torch, librosa
        w = librosa.resample(np.asarray(audio, np.float32), orig_sr=sr, target_sr=16000) if sr != 16000 else np.asarray(audio, np.float32)
        with torch.no_grad():
            enc, _ = self.m.forward(input_signal=torch.tensor(w)[None], input_signal_length=torch.tensor([len(w)]))
            lp = self.m.ctc_decoder(encoder_output=enc)[0].numpy()
        P = lp[:, [BL]+list(range(OFF, OFF+V))]
        P = P - np.logaddexp.reduce(P, axis=1, keepdims=True)
        return P, len(w)/16/len(P)

    def hear(self, audio, sr):
        "What Su-śrotā hears: greedy CTC decode."
        ids, out, prev = self.post(audio, sr)[0].argmax(1), [], -1
        for i in ids:
            if i != prev and i: out.append(self.tok.ids_to_tokens([int(i)-1])[0])
            prev = i
        return norm(''.join(out).replace('▁', ' '))

## Best of a few seeds

A render whose transcript is far from its text is retried with the next seed; the closest is kept. `Recitation.cer` records how far it was.

In [ ]:
#| export
def best_of(render, ear, text, tries=3, max_cer=0.08, seed=60):
    "Call `render(seed=…)` up to `tries` times, keeping the render closest to `text`."
    best = None
    for k in range(tries):
        r = render(seed=seed+7*k)
        r.cer = cer(text, ear.hear(r.audio, r.sr))
        if best is None or r.cer < best.cer: best = r
        if best.cer <= max_cer: break
    return best

## Word timings

Each piece's words are force-aligned within that piece's span, so one bad line cannot drag the rest. Words are given as displayed; each is
tokenised from its cleaned form, and a word with no Sanskrit tokens (a lone daṇḍa) gets an empty span at its neighbour's end.

In [ ]:
#| export
@patch
def words(self:Ear, rec, words):
    "For each piece of `rec` and its list of displayed `words`, `[word, start_ms, end_ms]` per word."
    import torch, torchaudio.functional as F
    P, fms = self.post(rec.audio, rec.sr)
    res = []
    for (s,e), ws in zip(rec.spans, words):
        a, b = int(s/fms), min(len(P), int(np.ceil(e/fms)))
        ids = [[i+1 for i in self.tok.text_to_ids(norm(clean(w)))] for w in ws]
        flat = [i for o in ids for i in o]
        if not flat or b-a < len(flat): res.append([[w, s, e] for w in ws]); continue
        ali, sc = F.forced_align(torch.tensor(np.ascontiguousarray(P[a:b]))[None], torch.tensor([flat]), blank=0)
        toks = F.merge_tokens(ali[0], sc[0].exp())
        out, k, end = [], 0, s
        for w,o in zip(ws, ids):
            if not o: out.append([w, end, end]); continue
            t0, t1 = toks[k], toks[k+len(o)-1]; k += len(o)
            out.append([w, s+round(t0.start*fms), s+round(t1.end*fms)]); end = out[-1][2]
        for i in range(len(out)-1): out[i][2] = max(out[i][2], out[i+1][1])
        res.append(out)
    return res

These need the Su-śrotā checkpoint (520 MB) and the Vāgdhenu weights, so they do not run in the tests.

In [ ]:
#| eval: false
from vachaka.core import Chanter
ch, ear = Chanter(), Ear()
lines = ['वसुदेवसुतं देवं कंसचाणूरमर्दनम्', 'देवकीपरमानन्दं कृष्णं वन्दे जगद्गुरुम्']
r = best_of(partial(ch.chant, lines), ear, ' '.join(lines))
r.cer, ear.words(r, [l.split() for l in lines])

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()